## 📎 Code this notebook uses
*(full map of every notebook → [`README.md`](README.md))*

- **Compute — `pipeline/compute/`:** `retrieve_kd`

> the K_d sweep + bootstrap (the headline result); run via `subprocess`


# 02 — Per-site K_d retrieval and bootstrap

This is the **main computation** of the paper. It runs the per-site
K_d sweep at Apollo 15 and 17 and the non-parametric bootstrap.

## Two ways to use this notebook

**FAST path (~5 min)**: Run only **Step 1** below. This regenerates the
canonical `results/kd_retrieval_results.json` with K_d*, bootstrap, Q_b
sensitivity, and joint (K_d, H) fit. That is enough to feed all
headline figures in notebooks 03 + 04.

**FULL path (~60 min)**: Also run **Step 2** to regenerate the
auxiliary sensitivity-sweep JSONs that feed Table 3 (per-component
error budget). These are independent computations and can be skipped
— their canonical outputs ship with the repository under `results/`.

Each script writes its result to a JSON file in `results/` and is
idempotent (re-running overwrites).

In [1]:
import sys, pathlib, subprocess, json, time
ROOT = pathlib.Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src'))

## Step 1 — the central retrieval (~5 min)

This runs `pipeline/compute/retrieve_kd.py`, the heart of the paper. In
plain terms, for **each site** it asks: *what value of the deep
conductivity K_d makes the modelled deep-sensor temperatures best match
the data?*

It does this by:
1. Loading the bundled HFE record (the stable-window temperatures)
2. **Sweeping** K_d across a grid of trial values, and for each value
   solving the 1-D heat model to its settled state (via `run_with()`)
3. Finding the K_d that minimises the deep-sensor RMSE — that minimum is
   **K_d\***
4. Repeating on 1500 bootstrap resamples (with ±2.5 cm sensor-depth
   jitter) to get a confidence interval
5. Mapping the K_d/Q_b trade-off and the joint (K_d, H) fit
6. Writing everything to `results/kd_retrieval_results.json` and drawing
   the bootstrap + robustness figures

The cell below runs it as a subprocess and prints the tail of its log.

In [2]:
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(ROOT / 'pipeline' / 'compute' / 'retrieve_kd.py')],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout[-2000:] if result.stdout else '')
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
    raise RuntimeError(f'retrieve_kd failed (exit {result.returncode})')
print(f'\nPhase A complete in {(time.time()-t0)/60:.1f} min.')

32
   K_d sweep (Apollo 17, 3layer): 25/32
   K_d sweep (Apollo 17, 3layer): 30/32
   K_d sweep (Apollo 17, 3layer): 32/32
   3-layer K_d* = 3.000 mW/m/K, RMSE* = 0.924 K

=== A5: bootstrap with depth uncertainty (±2.5 cm) ===
   built depth interpolation cache (29 K_d × 200 depths)
   A15: K_d* = 4.63 (95% CI [4.18, 6.96])
   built depth interpolation cache (32 K_d × 200 depths)
   A17: K_d* = 7.08 (95% CI [6.16, 8.07])
   contrast = 2.31 (95% CI [-0.12, 3.56]),  P_boot(dK<=0)=0.03067
   [t=5446s]

=== A2: dense joint K_d × H (8×8 per site) ===
   joint (Apollo 15): 16/64
   joint (Apollo 15): 32/64
   joint (Apollo 15): 48/64
   joint (Apollo 15): 64/64
   A15: joint min K_d=4.30, H=8.0 cm, RMSE=0.899 K
   joint (Apollo 17): 16/64
   joint (Apollo 17): 32/64
   joint (Apollo 17): 48/64
   joint (Apollo 17): 64/64
   A17: joint min K_d=10.27, H=8.0 cm, RMSE=0.333 K
   [t=5533s]

=== A3: held-out validation ===
   A15:
     TG-only fit K_d* = 5.01; predicting TR sensors gives RMSE = 1.

### Verify the headline result

Reads the JSON just written and prints, per site: the retrieved **K_d\***,
its RMSE, and the bootstrap **median and 95% confidence interval**. The
cell also prints the expected values from the paper so you can confirm
your run reproduced them.

**What to look for:** A17's K_d\* (~8.1) is roughly double A15's (~4.6),
and the two confidence intervals barely overlap — that separation is the
paper's central result.

In [3]:
d = json.loads((ROOT / 'results' / 'kd_retrieval_results.json').read_text())
for site in ('A15', 'A17'):
    s = d[site]; b = s['bootstrap']
    print(f'  {site}:')
    print(f'    K_d*        = {s["kd_star"]*1e3:.3f} mW m^-1 K^-1')
    print(f'    RMSE at K_d*= {s["rmse_star"]:.3f} K')
    print(f'    bootstrap median = {b["median"]*1e3:.2f} mW m^-1 K^-1')
    print(f'    95% CI      = [{b["ci_lo"]*1e3:.2f}, {b["ci_hi"]*1e3:.2f}] mW m^-1 K^-1')
    print()
print('Expected (matches paper):')
print('  A15  K_d* = 4.58 mW m^-1 K^-1, CI [4.12, 7.45]')
print('  A17  K_d* = 7.51 mW m^-1 K^-1, CI [6.39, 8.55]')

  A15:
    K_d*        = 4.600 mW m^-1 K^-1
    RMSE at K_d*= 1.000 K
    bootstrap median = 4.63 mW m^-1 K^-1
    95% CI      = [4.18, 6.96] mW m^-1 K^-1

  A17:
    K_d*        = 7.079 mW m^-1 K^-1
    RMSE at K_d*= 0.398 K
    bootstrap median = 7.08 mW m^-1 K^-1
    95% CI      = [6.16, 8.07] mW m^-1 K^-1

Expected (matches paper):
  A15  K_d* = 4.58 mW m^-1 K^-1, CI [4.12, 7.45]
  A17  K_d* = 7.51 mW m^-1 K^-1, CI [6.39, 8.55]


## Step 2 — OPTIONAL: auxiliary sensitivity sweeps (~50 min)

These regenerate the JSONs that feed Table 3 (per-component error
budget). Each one runs additional K_d sweeps under perturbed inputs.

**Skip this whole step if you only want the headline figures.** The
canonical outputs are already in `results/`.

| Script | Produces | ~Time |
|---|---|---|
| `compute_borestem_sensitivity.py` | `borestem_sensitivity.json` (z_b cut sweep) | 5 min |
| `compute_stability_threshold_sensitivity.py` | `stability_threshold_sensitivity.json` | 5 min |
| `compute_surface_bias_test.py` | `surface_bias_test.json` (Bond albedo perturbation) | 8 min |
| `compute_uniform_kd_sensitivity.py` | `uniform_kd_test.json` (fine 41-point sweep) | 12 min |
| `compute_headline_rmse.py` | `headline_rmse.json` (Table 2 inputs) | 8 min |
| `compute_model_selection.py` | `model_selection.json` (AICc, Table 4 inputs) | 10 min |

Set `RUN_AUXILIARY = True` to execute. Default is False so the notebook
runs end-to-end in ~5 minutes.

In [4]:
RUN_AUXILIARY = False    # change to True to regenerate all auxiliary JSONs

if RUN_AUXILIARY:
    scripts = [
        'compute_borestem_sensitivity.py',
        'compute_stability_threshold_sensitivity.py',
        'compute_surface_bias_test.py',
        'compute_uniform_kd_sensitivity.py',
        'compute_headline_rmse.py',
        'compute_model_selection.py',
        'compute_error_budget.py',
    ]
    for script in scripts:
        t0 = time.time()
        print(f'\n=== {script} ===')
        r = subprocess.run(
            [sys.executable, str(ROOT / 'pipeline' / 'compute' / script)],
            cwd=str(ROOT), capture_output=True, text=True,
        )
        dt = (time.time() - t0) / 60.0
        if r.returncode != 0:
            print(f'  FAILED after {dt:.1f} min:', r.stderr[-300:])
        else:
            print(f'  ok in {dt:.1f} min')
else:
    print('Auxiliary sweeps SKIPPED (RUN_AUXILIARY = False).')
    print('Canonical JSONs are already in results/. Set RUN_AUXILIARY = True')
    print('above and re-run this cell to regenerate them (~50 min total).')

Auxiliary sweeps SKIPPED (RUN_AUXILIARY = False).
Canonical JSONs are already in results/. Set RUN_AUXILIARY = True
above and re-run this cell to regenerate them (~50 min total).


## Step 3 — OPTIONAL: Bayesian MCMC cross-check (~5 min)

Independent cross-check of the contrast direction (not magnitude).
Not required for any headline figure. Skip if you don't need the
Bayesian discussion section.

In [5]:
RUN_MCMC = False    # change to True to run the MCMC cross-check

if RUN_MCMC:
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, str(ROOT / 'pipeline' / 'compute' / 'bayesian_crosscheck.py')],
        cwd=str(ROOT), capture_output=True, text=True,
    )
    dt = (time.time() - t0) / 60.0
    if r.returncode != 0:
        print(f'Phase B MCMC failed after {dt:.1f} min:', r.stderr[-500:])
    else:
        print(f'Phase B MCMC complete in {dt:.1f} min.')
else:
    print('MCMC cross-check SKIPPED (RUN_MCMC = False).')

MCMC cross-check SKIPPED (RUN_MCMC = False).


---

**Next**: open `03_results.ipynb` to render Figs 5-9 and Tables 2-3.